install the required libraries

In [ ]:
!pip install --upgrade pip
!pip install yfinance pandas numpy
!pip install pyarrow

Download the S&P company list 

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Create a browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
}

import os

# Reuse the saved snapshot (retrieved 10 November 2026) so results are reproducible.
# Delete sp500_company_list.csv to download the list again.
if os.path.exists("sp500_company_list.csv"):
    sp500 = pd.read_csv("sp500_company_list.csv")
    print("Loaded saved constituent snapshot (retrieved 10 November 2026)")
else:
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    tables = pd.read_html(StringIO(response.text))
    sp500 = tables[0]

# Display first 10 companies
display(sp500.head(10))

# Count companies
print("Number of companies:", len(sp500))

if not os.path.exists("sp500_company_list.csv"):
    sp500.to_csv("sp500_company_list.csv", index=False)

print("Company list saved successfully!")

In [ ]:
tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()

print("Number of tickers:", len(tickers))
print(tickers[:10])

In [ ]:
import yfinance as yf
import os
import time

# Create folder for downloaded data
os.makedirs("stock_data", exist_ok=True)

# Download in batches of 50
batch_size = 50

for i in range(0, len(tickers), batch_size):

    batch = tickers[i:i + batch_size]

    # Reuse saved snapshot if this batch was already downloaded
    if os.path.exists(f"stock_data/batch_{i // batch_size + 1}.parquet"):
        print("Using saved batch:", i // batch_size + 1)
        continue

    print("Downloading batch:", i // batch_size + 1)

    data = yf.download(
        tickers=batch,
        start="2021-01-01",
        end="2025-01-01",
        interval="1d",
        auto_adjust=False,
        actions=True,
        group_by="ticker",
        threads=True,
        progress=False
    )

    # Save each batch
    data.to_parquet(
        f"stock_data/batch_{i // batch_size + 1}.parquet"
    )

    time.sleep(2)

print("Download process finished!")

In [ ]:
import yfinance as yf

failed_tickers = ["FDXF", "HONA", "Q", "SNDK", "VYLR"]

for ticker in failed_tickers:

    print("\nChecking:", ticker)

    data = yf.download(
        ticker,
        start="2021-01-01",
        end="2025-01-01",
        auto_adjust=False,
        progress=False
    )

    if data.empty:
        print("No historical data available.")
    else:
        print("Data available!")
        print("First date:", data.index.min())
        print("Last date:", data.index.max())
        print("Number of records:", len(data))

In [ ]:
import pandas as pd

excluded = pd.DataFrame({
    "Ticker": ["FDXF", "HONA", "Q", "SNDK", "VYLR"],
    "Reason": [
        "No historical data available from Yahoo Finance",
        "No historical data available from Yahoo Finance",
        "No historical data available from Yahoo Finance",
        "No historical data available from Yahoo Finance",
        "No historical data available from Yahoo Finance"
    ]
})

excluded.to_csv("excluded_tickers.csv", index=False)
display(excluded)

In [ ]:
import glob
import os

files = glob.glob("stock_data/*.parquet")

print("Number of saved batch files:", len(files))

for file in files:
    print(os.path.basename(file),
          round(os.path.getsize(file) / 1024, 2), "KB")

In [ ]:
import pandas as pd
import glob
import os

files = sorted(glob.glob("stock_data/*.parquet"))

# Open the first batch
sample = pd.read_parquet(files[0])

print("File inspected:", os.path.basename(files[0]))
print("\nData dimensions:")
print(sample.shape)

print("\nFirst five rows:")
display(sample.head())

print("\nColumn structure:")
print(sample.columns)

Combine all 11 batches 

In [ ]:
import pandas as pd
import glob
import os

# Find all batch files and sort them numerically
files = glob.glob("stock_data/batch_*.parquet")
files = sorted(files, key=lambda x: int(
    os.path.basename(x).replace("batch_", "").replace(".parquet", "")
))

# Read all batches
batch_data = []

for file in files:
    data = pd.read_parquet(file)
    batch_data.append(data)
    print(os.path.basename(file), data.shape)

# Combine all companies into one dataset
all_stocks = pd.concat(batch_data, axis=1)

# Sort dates
all_stocks = all_stocks.sort_index()

# Save the combined dataset
all_stocks.to_parquet("all_stocks_combined.parquet")

print("\nCombined dataset dimensions:", all_stocks.shape)
print("First date:", all_stocks.index.min())
print("Last date:", all_stocks.index.max())
print("Number of unique tickers:",
      all_stocks.columns.get_level_values("Ticker").nunique())

# Check historical data coverage 

In [ ]:
# Get all unique company tickers
tickers_in_data = all_stocks.columns.get_level_values("Ticker").unique()

coverage = []

for ticker in tickers_in_data:

    # Get this company's data
    company_data = all_stocks[ticker]

    # Count available closing prices and volume observations
    close_count = company_data["Close"].notna().sum()
    volume_count = company_data["Volume"].notna().sum()

    # Find first and last available closing-price dates
    valid_dates = company_data.index[company_data["Close"].notna()]

    first_date = valid_dates.min() if len(valid_dates) > 0 else None
    last_date = valid_dates.max() if len(valid_dates) > 0 else None

    coverage.append({
        "Ticker": ticker,
        "Close observations": close_count,
        "Volume observations": volume_count,
        "First available date": first_date,
        "Last available date": last_date
    })

coverage_df = pd.DataFrame(coverage)

# Show companies with the fewest closing-price observations
coverage_df = coverage_df.sort_values("Close observations")

display(coverage_df.head(20))

print("\nTotal companies:", len(coverage_df))
print("Companies with no closing prices:",
      (coverage_df["Close observations"] == 0).sum())

# Identify Companies with full period data 

In [ ]:
# Expected number of trading dates in the downloaded dataset
expected_days = len(all_stocks.index)

# Add a percentage coverage column
coverage_df["Coverage %"] = (
    coverage_df["Close observations"] / expected_days * 100
).round(2)

# Identify companies with data on every trading date
coverage_df["Full history"] = (
    coverage_df["Close observations"] == expected_days
)

# Summarise the results
print("Expected trading dates:", expected_days)

print("Companies with full history:",
      coverage_df["Full history"].sum())

print("Companies without full history:",
      (~coverage_df["Full history"]).sum())

print("\nCompanies with full history:")
display(
    coverage_df[coverage_df["Full history"]]
    .sort_values("Ticker")
)

print("\nCompanies without full history:")
display(
    coverage_df[~coverage_df["Full history"]]
    .sort_values("Close observations")
)

In [ ]:
coverage_df.to_csv("stock_data_coverage_report.csv", index=False)

print("Coverage report saved successfully.")

# Creating clean Dataset 

In [ ]:
# Get the tickers with complete historical data
eligible_tickers = coverage_df.loc[
    coverage_df["Full history"], "Ticker"
].tolist()

# Keep only these companies in the combined dataset
clean_stocks = all_stocks.loc[
    :, all_stocks.columns.get_level_values("Ticker").isin(eligible_tickers)
].copy()

# Save the clean dataset
clean_stocks.to_parquet("clean_stocks_488.parquet")

# Save the list of included companies
pd.DataFrame({"Ticker": eligible_tickers}).to_csv(
    "included_tickers.csv", index=False
)

# Save the list of excluded companies
coverage_df.loc[
    ~coverage_df["Full history"]
].to_csv("excluded_tickers_full_report.csv", index=False)

print("Clean dataset dimensions:", clean_stocks.shape)
print("Number of included companies:",
      clean_stocks.columns.get_level_values("Ticker").nunique())
print("First date:", clean_stocks.index.min())
print("Last date:", clean_stocks.index.max())

Download the S&P 500 benchmark

In [ ]:
import yfinance as yf
import os
import pandas as pd

# Reuse the saved benchmark snapshot so results are reproducible.
# Delete sp500_benchmark.parquet to download ^GSPC again.
if os.path.exists("sp500_benchmark.parquet"):
    benchmark = pd.read_parquet("sp500_benchmark.parquet")
    print("Loaded saved benchmark snapshot")
else:
    benchmark = yf.download(
        "^GSPC",
        start="2021-01-01",
        end="2025-01-01",
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False
    )
    benchmark.to_parquet("sp500_benchmark.parquet")
    print("Benchmark saved successfully.")

print("Benchmark dimensions:", benchmark.shape)
print("First date:", benchmark.index.min())
print("Last date:", benchmark.index.max())

display(benchmark.head())
print(benchmark.columns)


# Extract adjusted closing pricess 

In [ ]:
# Extract adjusted closing prices for all 488 companies
stock_adj = clean_stocks.xs(
    "Adj Close",
    axis=1,
    level="Price"
)

# Extract adjusted closing price for the S&P 500
benchmark_adj = benchmark.xs(
    "Adj Close",
    axis=1,
    level="Price"
).iloc[:, 0]

print("Stock prices shape:", stock_adj.shape)
print("Benchmark prices shape:", benchmark_adj.shape)

display(stock_adj.head())
display(benchmark_adj.head())

#Convert daily prices into weekly prices
# Get the last available adjusted closing price in each Friday-ending week
weekly_stock_prices = stock_adj.resample("W-FRI").last()
weekly_benchmark_prices = benchmark_adj.resample("W-FRI").last()

# drop final incomplete week
weekly_stock_prices = weekly_stock_prices.loc[:"2024-12-27"]
weekly_benchmark_prices = weekly_benchmark_prices.loc[:"2024-12-27"]

print("Weekly stock prices:", weekly_stock_prices.shape)
print("Weekly benchmark prices:", weekly_benchmark_prices.shape)

display(weekly_stock_prices.head())
display(weekly_benchmark_prices.head())

# Calculate weekly percentage returns
weekly_stock_returns = weekly_stock_prices.pct_change()
weekly_benchmark_returns = weekly_benchmark_prices.pct_change()

# Remove the first week because it has no previous week for comparison
weekly_stock_returns = weekly_stock_returns.dropna()
weekly_benchmark_returns = weekly_benchmark_returns.dropna()

print("Weekly stock returns:", weekly_stock_returns.shape)
print("Weekly benchmark returns:", weekly_benchmark_returns.shape)

display(weekly_stock_returns.head())
display(weekly_benchmark_returns.head())

#Calculate mean weekly return and beta
# Create a results table
risk_return = pd.DataFrame(index=weekly_stock_returns.columns)


# Arithmetic mean of weekly returns
risk_return["Mean_Weekly_Return"] = (
    weekly_stock_returns.mean()
)

# Calculate beta for each company
market_variance = weekly_benchmark_returns.var()

for ticker in weekly_stock_returns.columns:

    covariance = weekly_stock_returns[ticker].cov(
        weekly_benchmark_returns
    )

    risk_return.loc[ticker, "Beta"] = (
        covariance / market_variance
    )

# Display results
risk_return.index.name = "Ticker"

print("Risk-return dataset dimensions:", risk_return.shape)

display(risk_return.head(10))

# Save results
risk_return.to_csv("risk_return_features.csv")
print("Risk-return features saved successfully.")

# Run a quality check on risk return Dataset 

In [ ]:
import numpy as np
import pandas as pd

print("Missing values:")
print(risk_return.isna().sum())

print("\nSummary statistics:")
display(risk_return.describe())

print("\nCompanies with missing values:")
display(risk_return[risk_return.isna().any(axis=1)])

# Get the latest 60 trading dates
latest_60_dates = clean_stocks.index[-60:]

# Identify the final observation date
observation_date = clean_stocks.index.max()

print("Observation date:", observation_date)
print("First date in latest 60 sessions:", latest_60_dates[0])
print("Last date in latest 60 sessions:", latest_60_dates[-1])
print("Number of sessions:", len(latest_60_dates))

rfm_results = []

for ticker in eligible_tickers:

    # Extract daily data for this company
    company = clean_stocks[ticker]

    close = company["Close"]
    volume = company["Volume"]

    # Average volume over the preceding 20 sessions
    previous_20_average = (
        volume.shift(1).rolling(window=20).mean()
    )

    # Identify unusually high-volume days
    high_volume_days = volume > (2 * previous_20_average)

    # RECENCY: latest qualifying high-volume date
    qualifying_dates = volume.index[high_volume_days.fillna(False)]

    if len(qualifying_dates) > 0:
        latest_high_volume_date = qualifying_dates.max()

        recency = (
            observation_date - latest_high_volume_date
        ).days
    else:
        recency = np.nan

    # FREQUENCY: number of qualifying days in latest 60 sessions
    frequency = high_volume_days.loc[latest_60_dates].sum()

    # MONETARY: sum of Close multiplied by Volume
    monetary = (
        close.loc[latest_60_dates] *
        volume.loc[latest_60_dates]
    ).sum()

    # Store results
    rfm_results.append({
        "Ticker": ticker,
        "Recency": recency,
        "Frequency": frequency,
        "Monetary": monetary
    })

# Create RFM dataset
rfm = pd.DataFrame(rfm_results).set_index("Ticker")

print("RFM dataset dimensions:", rfm.shape)
display(rfm.head(10))

rfm.to_csv("rfm_features.csv")
print("RFM dataset saved successfully.")

Validate RFM results 

In [ ]:
print("Missing values:")
print(rfm.isna().sum())

print("\nRFM summary statistics:")
display(rfm.describe())

print("\nNegative values:")
print((rfm < 0).sum())

print("\nCompanies with missing RFM values:")
display(rfm[rfm.isna().any(axis=1)])

print("\nFrequency range:")
print(rfm["Frequency"].min(), "to", rfm["Frequency"].max())

print("\nRecency range:")
print(rfm["Recency"].min(), "to", rfm["Recency"].max())

Prepare the Dataset for the clustering 

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# --------------------------------
# A. Prepare risk-return dataset
# --------------------------------

risk_return_clean = risk_return.copy()

risk_scaler = StandardScaler()

risk_scaled_array = risk_scaler.fit_transform(
    risk_return_clean
)

risk_scaled = pd.DataFrame(
    risk_scaled_array,
    index=risk_return_clean.index,
    columns=risk_return_clean.columns
)

# --------------------------------
# B. Prepare RFM dataset
# --------------------------------

rfm_clean = rfm.copy()

# Log transformation of Monetary
rfm_clean["Monetary_log"] = np.log1p(
    rfm_clean["Monetary"]
)

# Use Recency, Frequency and transformed Monetary
rfm_features = rfm_clean[
    ["Recency", "Frequency", "Monetary_log"]
]

# Standardise all three features
rfm_scaler = StandardScaler()

rfm_scaled_array = rfm_scaler.fit_transform(
    rfm_features
)

rfm_scaled = pd.DataFrame(
    rfm_scaled_array,
    index=rfm_features.index,
    columns=rfm_features.columns
)

print("Risk-return scaled shape:", risk_scaled.shape)
print("RFM scaled shape:", rfm_scaled.shape)

print("\nRisk-return scaled data:")
display(risk_scaled.head())

print("\nRFM scaled data:")
display(rfm_scaled.head())

risk_scaled.to_csv("risk_return_scaled.csv")
rfm_scaled.to_csv("rfm_scaled.csv")

print("Both scaled datasets saved successfully.")

# Hierarchical clustering 
Risk return Dataset

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

risk_linkage = linkage(
    risk_scaled,
    method="ward",
    metric="euclidean"
)

# Plot a readable dendrogram
plt.figure(figsize=(16, 7))

dendrogram(
    risk_linkage,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True
)

plt.title("Hierarchical Clustering Dendrogram - Risk-Return")
plt.xlabel("Cluster")
plt.ylabel("Ward Distance")
plt.tight_layout()
plt.show()

RFM

In [ ]:
rfm_linkage = linkage(
    rfm_scaled,
    method="ward",
    metric="euclidean"
)

plt.figure(figsize=(16, 7))

dendrogram(
    rfm_linkage,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True
)

plt.title("Hierarchical Clustering Dendrogram - RFM")
plt.xlabel("Cluster")
plt.ylabel("Ward Distance")
plt.tight_layout()
plt.show()

Test different number of clusters 

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

risk_silhouette = []

for k in range(2, 9):
    model = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )

    labels = model.fit_predict(risk_scaled)

    score = silhouette_score(risk_scaled, labels)

    risk_silhouette.append({
        "Number_of_Clusters": k,
        "Silhouette_Score": score
    })

risk_silhouette_df = pd.DataFrame(risk_silhouette)

display(risk_silhouette_df)

In [ ]:
rfm_silhouette = []

for k in range(2, 9):
    model = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )

    labels = model.fit_predict(rfm_scaled)

    score = silhouette_score(rfm_scaled, labels)

    rfm_silhouette.append({
        "Number_of_Clusters": k,
        "Silhouette_Score": score
    })

rfm_silhouette_df = pd.DataFrame(rfm_silhouette)

display(rfm_silhouette_df)

Save the hierarchical clustering results 

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Final risk-return hierarchical clustering
risk_hierarchical = AgglomerativeClustering(
    n_clusters=8,
    linkage="ward"
)

risk_hierarchical_labels = risk_hierarchical.fit_predict(risk_scaled)

# Final RFM hierarchical clustering
rfm_hierarchical = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward"
)

rfm_hierarchical_labels = rfm_hierarchical.fit_predict(rfm_scaled)

# Store cluster labels
hierarchical_results = pd.DataFrame(index=risk_scaled.index)

hierarchical_results["Risk_Return_Cluster"] = risk_hierarchical_labels
# rfm_scaled rows are in a different ticker order from risk_scaled, so align RFM labels by ticker
hierarchical_results["RFM_Cluster"] = pd.Series(rfm_hierarchical_labels, index=rfm_scaled.index)

display(hierarchical_results.head())

# Save results
hierarchical_results.to_csv("hierarchical_clustering_results.csv")

Examine the cluster sizes 

In [ ]:
print("Risk-return cluster sizes:")
print(hierarchical_results["Risk_Return_Cluster"].value_counts().sort_index())

print("\nRFM cluster sizes:")
print(hierarchical_results["RFM_Cluster"].value_counts().sort_index())

K-means for risk return 

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

risk_kmeans_results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(risk_scaled)

    silhouette = silhouette_score(risk_scaled, labels)

    risk_kmeans_results.append({
        "Number_of_Clusters": k,
        "Silhouette_Score": silhouette,
        "Inertia": model.inertia_
    })

risk_kmeans_df = pd.DataFrame(risk_kmeans_results)

display(risk_kmeans_df)

K-means for RFM

In [ ]:
rfm_kmeans_results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(rfm_scaled)

    silhouette = silhouette_score(rfm_scaled, labels)

    rfm_kmeans_results.append({
        "Number_of_Clusters": k,
        "Silhouette_Score": silhouette,
        "Inertia": model.inertia_
    })

rfm_kmeans_df = pd.DataFrame(rfm_kmeans_results)

display(rfm_kmeans_df)

Plot the K-means result 

In [ ]:
import matplotlib.pyplot as plt

# Risk-return silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(
    risk_kmeans_df["Number_of_Clusters"],
    risk_kmeans_df["Silhouette_Score"],
    marker="o"
)
plt.title("K-means Silhouette Scores - Risk-Return")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 9))
plt.grid(True)
plt.show()

# Risk-return inertia
plt.figure(figsize=(8, 5))
plt.plot(
    risk_kmeans_df["Number_of_Clusters"],
    risk_kmeans_df["Inertia"],
    marker="o"
)
plt.title("K-means Inertia - Risk-Return")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(range(2, 9))
plt.grid(True)
plt.show()

# RFM silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(
    rfm_kmeans_df["Number_of_Clusters"],
    rfm_kmeans_df["Silhouette_Score"],
    marker="o"
)
plt.title("K-means Silhouette Scores - RFM")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 9))
plt.grid(True)
plt.show()

# RFM inertia
plt.figure(figsize=(8, 5))
plt.plot(
    rfm_kmeans_df["Number_of_Clusters"],
    rfm_kmeans_df["Inertia"],
    marker="o"
)
plt.title("K-means Inertia - RFM")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(range(2, 9))
plt.grid(True)
plt.show()

# Save the results
risk_kmeans_df.to_csv("risk_return_kmeans_evaluation.csv", index=False)
rfm_kmeans_df.to_csv("rfm_kmeans_evaluation.csv", index=False)

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

# Final risk-return K-means model
risk_kmeans_final = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

risk_kmeans_labels = risk_kmeans_final.fit_predict(risk_scaled)

# Final RFM K-means model
rfm_kmeans_final = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

rfm_kmeans_labels = rfm_kmeans_final.fit_predict(rfm_scaled)

# Combine all cluster assignments
final_results = pd.DataFrame(index=risk_scaled.index)

final_results["Risk_Return_Ward"] = risk_hierarchical_labels
final_results["Risk_Return_KMeans"] = risk_kmeans_labels
# align RFM labels by ticker (rfm_scaled has a different row order from risk_scaled)
final_results["RFM_Ward"] = pd.Series(rfm_hierarchical_labels, index=rfm_scaled.index)
final_results["RFM_KMeans"] = pd.Series(rfm_kmeans_labels, index=rfm_scaled.index)

display(final_results.head(10))

# Save results
final_results.to_csv("final_clustering_results.csv")

In [ ]:
print("Risk-return K-means cluster sizes:")
print(final_results["Risk_Return_KMeans"].value_counts().sort_index())

print("\nRFM K-means cluster sizes:")
print(final_results["RFM_KMeans"].value_counts().sort_index())

In [ ]:
# Risk-return cluster profiles
risk_profile = risk_return.join(
    final_results["Risk_Return_KMeans"]
).groupby("Risk_Return_KMeans").agg(
    Company_Count=("Mean_Weekly_Return", "count"),
    Average_Weekly_Return=("Mean_Weekly_Return", "mean"),
    Average_Beta=("Beta", "mean")
)

print("Risk-return K-means profiles:")
display(risk_profile.round(4))


# RFM cluster profiles
rfm_profile = rfm.join(
    final_results["RFM_KMeans"]
).groupby("RFM_KMeans").agg(
    Company_Count=("Recency", "count"),
    Average_Recency=("Recency", "mean"),
    Average_Frequency=("Frequency", "mean"),
    Average_Monetary=("Monetary", "mean")
)

print("RFM K-means profiles:")
display(rfm_profile.round(2))

Examine the hierarchical cluster profile 

In [ ]:
# Risk-return Ward cluster profiles
risk_ward_profile = risk_return.join(
    final_results["Risk_Return_Ward"]
).groupby("Risk_Return_Ward").agg(
    Company_Count=("Mean_Weekly_Return", "count"),
    Average_Weekly_Return=("Mean_Weekly_Return", "mean"),
    Average_Beta=("Beta", "mean")
)

print("Risk-return Ward profiles:")
display(risk_ward_profile.round(4))


# RFM Ward cluster profiles
rfm_ward_profile = rfm.join(
    final_results["RFM_Ward"]
).groupby("RFM_Ward").agg(
    Company_Count=("Recency", "count"),
    Average_Recency=("Recency", "mean"),
    Average_Frequency=("Frequency", "mean"),
    Average_Monetary=("Monetary", "mean")
)

print("RFM Ward profiles:")
display(rfm_ward_profile.round(2))

RFM sensitivity analysist - remove monetary 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

# Step 1: Keep Recency and Frequency only
rf_only = rfm[["Recency", "Frequency"]].copy()

# Step 2: Standardise the two remaining features
rf_scaler = StandardScaler()
rf_scaled = pd.DataFrame(
    rf_scaler.fit_transform(rf_only),
    index=rf_only.index,
    columns=rf_only.columns
)

# Step 3: Repeat Ward clustering with 2 clusters
rf_ward_model = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward"
)

rf_ward_labels = rf_ward_model.fit_predict(rf_scaled)

# Step 4: Compare with original RFM Ward labels
ari_score = adjusted_rand_score(
    rfm_hierarchical_labels,
    rf_ward_labels
)

print("Adjusted Rand Index:", round(ari_score, 4))

# Step 5: Check revised cluster sizes
print("\nCluster sizes without Monetary:")
print(pd.Series(rf_ward_labels).value_counts().sort_index())

Examine correlations between RFM features 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np
rfm_correlations = rfm.assign(log1p_M=np.log1p(rfm["Monetary"]))[["Recency", "Frequency", "log1p_M"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    rfm_correlations,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix of RFM Features (R, F, log1p(M))")
plt.tight_layout()
plt.show()

display(rfm_correlations.round(3))

In [ ]:
import platform, datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, scipy, yfinance
from scipy.stats import skew, kurtosis
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# ---------- Environment and dates ----------
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__, "| yfinance", yfinance.__version__)
print("Date this cell was run:", datetime.date.today())   # use the date you actually downloaded; note it in the report

In [ ]:
import pandas as pd

clean_stocks = pd.read_parquet("clean_stocks_488.parquet")

print("Dataset loaded successfully")
print("Shape:", clean_stocks.shape)
display(clean_stocks.head())

In [ ]:
# ---------- Data validation ----------
import pandas as pd

clean_stocks = pd.read_parquet("clean_stocks_488.parquet")

dup_dates = clean_stocks.index.duplicated().sum()
adj  = clean_stocks.xs("Adj Close", axis=1, level="Price")
cls  = clean_stocks.xs("Close",     axis=1, level="Price")
vol  = clean_stocks.xs("Volume",    axis=1, level="Price")
div  = clean_stocks.xs("Dividends", axis=1, level="Price")
spl  = clean_stocks.xs("Stock Splits", axis=1, level="Price")
print("Duplicate dates:", dup_dates)
print("Missing Close / Adj Close / Volume:", cls.isna().sum().sum(), adj.isna().sum().sum(), vol.isna().sum().sum())
print("Non-positive Close / Adj Close:", (cls <= 0).sum().sum(), (adj <= 0).sum().sum())
print("Zero or negative volume (share-days):", (vol <= 0).sum().sum())
print("Shares with zero-volume days:", list(vol.columns[(vol <= 0).any()]))
print("Shares with a stock split in window:", list(spl.columns[(spl > 0).any()]))
print("Dividend-paying shares:", int((div > 0).any().sum()), "| shares where Close != Adj Close on first date:",
      int((cls.iloc[0] - adj.iloc[0]).abs().gt(1e-6).sum()))
big_moves = (adj.pct_change().abs() > 0.5).sum()
print("Shares with a one-day |return| > 50% (possible unadjusted split):", list(big_moves[big_moves > 0].index))

In [ ]:
stock_adj = clean_stocks.xs(
    "Adj Close",
    axis=1,
    level="Price"
).copy()

benchmark_raw = pd.read_parquet("sp500_benchmark.parquet")

benchmark_adj = benchmark_raw[
    ("Adj Close", "^GSPC")
].copy()

print("Stock adjusted prices:", stock_adj.shape)
print("Benchmark observations:", benchmark_adj.shape)

# ---------- Exclude incomplete first/final weeks and recompute ----------
sessions_per_week = clean_stocks.index.to_series().resample("W-FRI").count()
print(sessions_per_week.head(2), sessions_per_week.tail(2), sep="\n")
complete_weeks = sessions_per_week.index[:-1] if sessions_per_week.iloc[-1] < 3 else sessions_per_week.index
w_px  = stock_adj.resample("W-FRI").last().loc[complete_weeks]
b_px  = benchmark_adj.resample("W-FRI").last().loc[complete_weeks]
w_ret = w_px.pct_change().dropna()
b_ret = b_px.pct_change().dropna()
assert (w_ret.index == b_ret.index).all()
print("Aligned weekly observations:", len(w_ret))
risk_return_fixed = pd.DataFrame({"Mean_Weekly_Return": w_ret.mean(),
                                  "Beta": w_ret.apply(lambda s: s.cov(b_ret)) / b_ret.var()})
print(risk_return_fixed.describe().round(4))
if "risk_return" in globals():
    print(
        "Max abs change in beta vs notebook version:",
        (risk_return_fixed["Beta"] - risk_return["Beta"]).abs().max()
    )
else:
    print("Original risk_return table is not in memory.")
    print("Using the corrected risk_return_fixed table.")

In [ ]:
observation_date = clean_stocks.index.max()

latest_60_dates = clean_stocks.index[-60:]

print("Observation date:", observation_date.date())
print("60-session window:", latest_60_dates[0].date(),
      "to", latest_60_dates[-1].date())
print("Number of sessions:", len(latest_60_dates))

# ---------- Worked RFM example ----------
t = "WFC"
v, c = clean_stocks[t]["Volume"], clean_stocks[t]["Close"]
base = v.shift(1).rolling(20).mean()
thr  = 2 * base
flag = v > thr
ex = pd.DataFrame({"Close": c, "Volume": v, "Mean prev 20": base, "Threshold": thr, "High volume": flag})
print(ex.loc[latest_60_dates].query("`High volume`").round(0))
last_evt = flag[flag].index.max()
print("Latest event:", last_evt.date(), "| Recency =", (observation_date - last_evt).days)
print("Frequency (60 sessions) =", int(flag.loc[latest_60_dates].sum()))
print("Monetary = sum(Close*Volume) =", f"{(c.loc[latest_60_dates]*v.loc[latest_60_dates]).sum():,.0f}")
print("Rows with no valid baseline (first 20 sessions):", int(base.isna().sum()))

In [ ]:
# ---------- Rebuild RFM features ----------

observation_date = clean_stocks.index.max()

latest_60_dates = clean_stocks.index[-60:]

rfm_rows = []

for ticker in stock_adj.columns:

    close = clean_stocks[(ticker, "Close")]  # Monetary uses Close x Volume (brief 1.4.3)
    volume = clean_stocks[(ticker, "Volume")]

    previous_20_avg = volume.shift(1).rolling(20).mean()
    high_volume = volume > (2 * previous_20_avg)

    event_dates = high_volume[high_volume].index

    if len(event_dates) > 0:
        last_event = event_dates.max()
        recency = (observation_date - last_event).days
    else:
        recency = np.nan

    frequency = high_volume.loc[latest_60_dates].sum()

    monetary = (
        close.loc[latest_60_dates] *
        volume.loc[latest_60_dates]
    ).sum()

    rfm_rows.append({
        "Ticker": ticker,
        "Recency": recency,
        "Frequency": frequency,
        "Monetary": monetary
    })

rfm = pd.DataFrame(rfm_rows).set_index("Ticker")

print("RFM shape:", rfm.shape)
rfm.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import skew, kurtosis

observation_date = clean_stocks.index.max()

latest_60_dates = clean_stocks.index[-60:]

rfm_rows = []

for ticker in stock_adj.columns:

    close = clean_stocks[(ticker, "Close")]  # Monetary uses Close x Volume (brief 1.4.3)
    volume = clean_stocks[(ticker, "Volume")]

    previous_20_avg = volume.shift(1).rolling(20).mean()
    high_volume = volume > (2 * previous_20_avg)

    event_dates = high_volume[high_volume].index

    if len(event_dates) > 0:
        last_event = event_dates.max()
        recency = (observation_date - last_event).days
    else:
        recency = np.nan

    frequency = high_volume.loc[latest_60_dates].sum()

    monetary = (
        close.loc[latest_60_dates] *
        volume.loc[latest_60_dates]
    ).sum()

    rfm_rows.append({
        "Ticker": ticker,
        "Recency": recency,
        "Frequency": frequency,
        "Monetary": monetary
    })

rfm = pd.DataFrame(rfm_rows).set_index("Ticker")

print("RFM shape:", rfm.shape)
rfm.head()

risk_return = risk_return_fixed.copy()

print(risk_return.shape)

feat = pd.concat([
    risk_return,
    rfm.assign(log1p_M=np.log1p(rfm["Monetary"]))
], axis=1)

summ = pd.DataFrame({
    "skew": feat.apply(skew),
    "excess_kurtosis": feat.apply(kurtosis),
    "max_abs_z": ((feat - feat.mean()) / feat.std()).abs().max()
})

print(summ.round(2))

# ---------- Skew and outliers ----------
feat = pd.concat([risk_return, rfm.assign(log1p_M=np.log1p(rfm["Monetary"]))], axis=1)
summ = pd.DataFrame({"skew": feat.apply(skew), "excess_kurtosis": feat.apply(kurtosis),
                     "max_abs_z": ((feat - feat.mean()) / feat.std()).abs().max()})
print(summ.round(2))
feat.hist(bins=30, figsize=(14, 7)); plt.tight_layout(); plt.show()
print("Share of shares with Recency == mode:", (rfm["Recency"] == rfm["Recency"].mode()[0]).mean().round(3),
      "| mode =", rfm["Recency"].mode()[0])

# ---------- Correlation using log1p(M) as specified ----------
rfm_l = rfm.assign(log1p_M=np.log1p(rfm["Monetary"]))[["Recency", "Frequency", "log1p_M"]]
print(rfm_l.corr().round(3)); print(rfm_l.corr(method="spearman").round(3))

In [ ]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score

# Rebuild scaled feature matrices 

from sklearn.preprocessing import StandardScaler

risk_scaled = StandardScaler().fit_transform(
    risk_return[["Mean_Weekly_Return", "Beta"]]
)

rfm_log = rfm.assign(
    log1p_M=np.log1p(rfm["Monetary"])
)[["Recency", "Frequency", "log1p_M"]]

rfm_scaled = StandardScaler().fit_transform(rfm_log)

print("Risk scaled shape:", risk_scaled.shape)
print("RFM scaled shape:", rfm_scaled.shape)


# Recreate selected cluster labels

final_results = {}

final_results["Risk_Return_Ward"] = AgglomerativeClustering(
    n_clusters=8, linkage="ward"
).fit_predict(risk_scaled)

final_results["Risk_Return_KMeans"] = KMeans(
    n_clusters=2, random_state=42, n_init=20
).fit_predict(risk_scaled)

final_results["RFM_Ward"] = AgglomerativeClustering(
    n_clusters=2, linkage="ward"
).fit_predict(rfm_scaled)

final_results["RFM_KMeans"] = KMeans(
    n_clusters=2, random_state=42, n_init=20
).fit_predict(rfm_scaled)


rows = []
for k in range(2, 9):
    for name, X in [("Risk-return", risk_scaled), ("RFM", rfm_scaled)]:
        w = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)
        km = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
        rows.append({"Matrix": name, "k": k, "ARI_Ward_vs_KMeans": adjusted_rand_score(w, km),
                     "Ward_sizes": sorted(np.bincount(w), reverse=True),
                     "KMeans_sizes": sorted(np.bincount(km), reverse=True)})
cmp_df = pd.DataFrame(rows); print(cmp_df.to_string())
print("Selected solutions:",
      "risk Ward(8) vs KMeans(2) ARI =", round(adjusted_rand_score(final_results["Risk_Return_Ward"], final_results["Risk_Return_KMeans"]), 3),
      "| RFM Ward(2) vs KMeans(2) ARI =", round(adjusted_rand_score(final_results["RFM_Ward"], final_results["RFM_KMeans"]), 3))

In [ ]:
from sklearn.decomposition import PCA

# ---------- Labelled 2-D visualisations ----------
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, col, ttl in [(ax[0], "Risk_Return_Ward", "Risk-return: Ward (k=8)"), (ax[1], "Risk_Return_KMeans", "Risk-return: K-means (k=2)")]:
    sc = a.scatter(risk_return["Beta"], risk_return["Mean_Weekly_Return"], c=final_results[col], cmap="tab10", s=18)
    a.set_xlabel("Weekly beta"); a.set_ylabel("Mean weekly return"); a.set_title(ttl)
    a.legend(*sc.legend_elements(), title="Cluster", fontsize=8)
plt.tight_layout(); plt.show()
pca = PCA(n_components=2, random_state=42).fit(rfm_scaled)
Z = pca.transform(rfm_scaled)
print("PCA explained variance (RFM):", pca.explained_variance_ratio_.round(3), "| loadings:\n",
      pd.DataFrame(
       pca.components_,
       columns=["Recency", "Frequency", "log1p_M"],
       index=["PC1", "PC2"]
      ).round(2))     
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, col, ttl in [(ax[0], "RFM_Ward", "RFM: Ward (k=2)"), (ax[1], "RFM_KMeans", "RFM: K-means (k=2)")]:
    sc = a.scatter(Z[:, 0], Z[:, 1], c=final_results[col], cmap="tab10", s=18)
    a.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})"); a.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})"); a.set_title(ttl)
    a.legend(*sc.legend_elements(), title="Cluster", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Profiles in original units, medians, and R+F-only comparison ----------

rfm_full = rfm.copy()
rfm_full["Daily_traded_value"] = rfm_full["Monetary"] / 60

for col in ["RFM_Ward", "RFM_KMeans"]:
    labels = pd.Series(
        final_results[col],
        index=rfm.index,
        name=col
    )

    print("\n", col)

    profile = rfm_full.join(labels).groupby(col).agg(
        n=("Recency", "size"),
        R_med=("Recency", "median"),
        F_med=("Frequency", "median"),
        M_med=("Monetary", "median"),
        DTV_med=("Daily_traded_value", "median")
    )

    print(profile.round(0))

In [ ]:
# ---------- Feature table with cluster assignments ----------
# Recreate R+F-only clustering labels
import os


from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
print(os.listdir())
rf_scaled = pd.DataFrame(
    StandardScaler().fit_transform(
        rfm[["Recency", "Frequency"]]
    ),
    index=rfm.index,
    columns=["Recency", "Frequency"]
)

rf_labels = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward"
).fit_predict(rf_scaled)

print("R+F labels created:", len(rf_labels))

cluster_table = pd.DataFrame(
    {
        name: labels
        for name, labels in final_results.items()
    },
    index=rfm.index
)

cluster_table["RFM_Ward_RF_only"] = rf_labels
sp500 = pd.read_csv("sp500_company_list.csv")
print(sp500.columns.tolist())
sp500.head()

sector_table = sp500.copy()

sector_table["Ticker"] = (
    sector_table["Symbol"]
    .str.replace(".", "-", regex=False)
)

sector_table = sector_table.set_index("Ticker")[
    ["Security", "GICS Sector"]
]

feature_table = (
    risk_return
    .join(rfm)
    .assign(log1p_Monetary=np.log1p(rfm["Monetary"]))
    .join(cluster_table)
    .join(sector_table)
)

feature_table.to_csv("feature_table_with_clusters.csv")

print("Feature table shape:", feature_table.shape)
feature_table.head()

In [ ]:
# ---------- Sector composition and risk-return vs RFM agreement ----------
sol = {"Risk-return Ward (k=8)": "Risk_Return_Ward", "Risk-return K-means (k=2)": "Risk_Return_KMeans",
       "RFM Ward (k=2)": "RFM_Ward", "RFM K-means (k=2)": "RFM_KMeans"}
base = feature_table["GICS Sector"].value_counts(normalize=True)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (ttl, col) in zip(axes.ravel(), sol.items()):
    ct = pd.crosstab(feature_table[col], feature_table["GICS Sector"], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=ax, legend=False, colormap="tab20")
    ax.set_title(ttl); ax.set_ylabel("Share of cluster"); ax.set_xlabel("Cluster")
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, fontsize=8); plt.tight_layout(rect=[0, 0.08, 1, 1]); plt.show()
for ttl, col in sol.items():
    lift = pd.crosstab(feature_table[col], feature_table["GICS Sector"], normalize="index").div(base)
    print(ttl, "- sector share relative to base rate (>1 = over-represented)"); print(lift.round(2).to_string())
print("ARI risk-return K-means vs RFM K-means:", round(adjusted_rand_score(final_results["Risk_Return_KMeans"], final_results["RFM_KMeans"]), 3))
print("ARI risk-return Ward(8) vs RFM Ward(2):", round(adjusted_rand_score(final_results["Risk_Return_Ward"], final_results["RFM_Ward"]), 3))

In [ ]:
# Re-run of the RFM correlation heatmap on the corrected RFM table (log1p(M), Monetary from Close)
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np
rfm_correlations = rfm.assign(log1p_M=np.log1p(rfm["Monetary"]))[["Recency", "Frequency", "log1p_M"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    rfm_correlations,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix of RFM Features (R, F, log1p(M))")
plt.tight_layout()
plt.show()

display(rfm_correlations.round(3))